# Tiempo total de audio segmentado
Calcula la duración total cubierta por los segmentos anotados en uno o varios `source_segments.json`.

In [5]:
import json
from datetime import timedelta
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists():
            return p
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / 'src'))

from kinai.models.video_annotation import VideoAnnotation

JSON_PATHS = [
    #REPO_ROOT / 'corpora' / 'base' / 'annotations' / 'source_segments.json',
    REPO_ROOT / 'corpora' / 'v2' / 'annotations' / 'source_segments.json',
]

def load_videos(paths):
    out = []
    for p in paths:
        p = Path(p)
        with open(p, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for v in data:
            out.append((p.name, VideoAnnotation.from_dict(v)))
    return out

videos = load_videos(JSON_PATHS)
print(f'Archivos: {len(JSON_PATHS)}  |  Videos: {len(videos)}')


Archivos: 1  |  Videos: 130


In [6]:
def format_td(td: timedelta) -> str:
    total = int(td.total_seconds())
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

def merged_duration(segments):
    intervals = sorted([(s.start_td, s.end_td) for s in segments], key=lambda x: x[0])
    merged = timedelta(0)
    if not intervals:
        return merged
    cur_s, cur_e = intervals[0]
    for s, e in intervals[1:]:
        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            merged += cur_e - cur_s
            cur_s, cur_e = s, e
    merged += cur_e - cur_s
    return merged

# Totales por archivo fuente
per_file = {}
for src, v in videos:
    raw = sum((s.end_td - s.start_td for s in v.segments), timedelta(0))
    merged = merged_duration(v.segments)
    agg = per_file.setdefault(src, {'videos': 0, 'segs': 0, 'raw': timedelta(0), 'merged': timedelta(0)})
    agg['videos'] += 1
    agg['segs'] += len(v.segments)
    agg['raw'] += raw
    agg['merged'] += merged

for src, agg in per_file.items():
    print(f"{src}: {agg['videos']} videos, {agg['segs']} segs, crudo={format_td(agg['raw'])}, sin solapes={format_td(agg['merged'])}")

source_segments.json: 130 videos, 912 segs, crudo=00:52:34, sin solapes=00:50:49


In [7]:
total_raw = sum((agg['raw'] for agg in per_file.values()), timedelta(0))
total_merged = sum((agg['merged'] for agg in per_file.values()), timedelta(0))
total_videos = sum(agg['videos'] for agg in per_file.values())
total_segs = sum(agg['segs'] for agg in per_file.values())

print(f'Videos totales: {total_videos}')
print(f'Segmentos totales: {total_segs}')
print(f'Duración total (suma cruda):  {format_td(total_raw)} ({total_raw.total_seconds():.1f} s)')
print(f'Duración total (sin solapes): {format_td(total_merged)} ({total_merged.total_seconds():.1f} s)')

Videos totales: 130
Segmentos totales: 912
Duración total (suma cruda):  00:52:34 (3154.0 s)
Duración total (sin solapes): 00:50:49 (3049.0 s)


In [8]:
# Detalle por video
for src, v in videos:
    dur = merged_duration(v.segments)
    print(f'{format_td(dur)}  |  {len(v.segments):4d} segs  |  [{src}] {v.title[:60]}')

00:02:16  |    32 segs  |  [source_segments.json] 7.-DIÁLOGO 1, Aprenda Maya (SALUDOS, despedidas; preguntas y
00:02:28  |    36 segs  |  [source_segments.json] 12. Aprenda Maya, frases 10
00:01:16  |    17 segs  |  [source_segments.json] 13. Dialogo 2, Aprenda Maya
00:01:03  |    14 segs  |  [source_segments.json] 14.-Aprenda Maya, Frases 11 (tal vez, allá; despedidas)
00:01:30  |    24 segs  |  [source_segments.json] 15.-Aprenda Maya, Frases 12 (VERBOS: bailar, acostumbrar, so
00:01:30  |    27 segs  |  [source_segments.json] 16.-Aprenda Maya, Frases 13 (VERBOS: venir, comprar; órdenes
00:01:43  |    32 segs  |  [source_segments.json] 17.-Aprenda Maya, Frases 14 (VERBOS dar; aunque, tuve, tuvim
00:01:16  |    26 segs  |  [source_segments.json] 18.-Aprenda Maya, Frases 15 ( aquí, así es, así se hace...) 
00:02:16  |    23 segs  |  [source_segments.json] 19.-DIÁLOGO 3, Aprenda Maya (VERBOS: vender, comprar, pedir,
00:00:49  |    18 segs  |  [source_segments.json] 20.-Aprenda Maya, Fras